# Módulo 06: Validación, selección y generalización

[Abrir en Colab](https://colab.research.google.com/github/sgevatschnaider/data-science-business-decisions/blob/main/notebooks/06-validacion-modelos.ipynb)

**Pregunta de decisión:** ¿El desempeño observado representa casos futuros o es una consecuencia del azar, el sobreajuste o una partición incorrecta?

**Autor:** Sergio Gevatschnaider


## Objetivos

- Separar entrenamiento, validación y test por su función.
- Elegir K-Fold, estratificación, grupos o cortes temporales.
- Comparar modelos contra baselines relevantes.
- Reportar distribución de métricas y no solo un promedio.

**Criterio de éxito:** el resultado debe cambiar o sostener una acción concreta, superar una referencia y declarar límites.


## 1. Entorno reproducible

Registramos versiones y semilla antes de producir evidencia. Ejecutá siempre **Runtime → Run all** en Colab.


In [ ]:
import platform
import sys

import matplotlib
import numpy as np
import pandas as pd
import sklearn

SEED = 42
np.random.seed(SEED)
print({
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "matplotlib": matplotlib.__version__,
    "seed": SEED,
})

## 2. Experimento base

El bloque siguiente construye una referencia mínima y verificable. No representa todavía la recomendación final.


In [ ]:
import numpy as np
from sklearn.datasets import make_regression
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold, cross_validate

X, y = make_regression(n_samples=300, n_features=8, noise=20, random_state=42)
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for name, model in {"baseline": DummyRegressor(), "ridge": Ridge(alpha=1)}.items():
    scores = cross_validate(model, X, y, cv=cv, scoring="neg_mean_absolute_error")
    mae = -scores["test_score"]
    print(name, {"media": mae.mean().round(2), "desvio": mae.std().round(2)})

## 3. Evidencia visual

Una visualización útil permite comparar, muestra unidades y deja visible la incertidumbre o variación relevante.


In [ ]:
import matplotlib.pyplot as plt

scores = cross_validate(Ridge(alpha=1), X, y, cv=cv, scoring="neg_mean_absolute_error")
mae_folds = -scores["test_score"]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, len(mae_folds) + 1), mae_folds, color="#0f766e")
ax.axhline(mae_folds.mean(), color="#b45309", linestyle="--", label="Media")
ax.set(xlabel="Fold", ylabel="MAE", title="Variabilidad de la evaluación")
ax.legend()
plt.tight_layout()

## 4. Comparación para decidir

Un scoring entrenado con operaciones de los mismos clientes en train y test parece excelente, pero falla con clientes nuevos.

La tabla fuerza una comparación entre alternativas, costos o criterios. Adaptala a las unidades del caso.


In [ ]:
pd.DataFrame({'criterio': ['MAE medio', 'desvío entre folds', 'latencia', 'valor'], 'ridge': [mae_folds.mean(), mae_folds.std(), 1, 4], 'baseline': [float(np.std(y)), 0, 1, 1]})

## 5. Desafío de transferencia

**Diseñá una validación que replique quién, cuándo y dónde recibirá predicciones en producción.**

1. Identificar dependencias temporales o por entidad.
2. Construir un baseline antes de optimizar.
3. Aplicar el esquema de validación dentro del pipeline.
4. Reportar media, dispersión y comparación con test.

Antes de continuar, escribí una hipótesis, una condición que la refutaría y el costo de una decisión equivocada.

### Registro de decisión

Completá la celda siguiente como evidencia de cierre del laboratorio.


In [ ]:
decision_record = {
    "pregunta": '¿El desempeño observado representa casos futuros o es una consecuencia del azar, el sobreajuste o una partición incorrecta?',
    "hipotesis": "Completar antes del análisis",
    "evidencia": "Registrar la tabla o visualización que cambia la decisión",
    "recomendacion": "Expresar acción, población y horizonte",
    "limitacion": "Indicar qué podría invalidar la conclusión",
    "responsable": "Asignar dueño y fecha de revisión",
}
pd.Series(decision_record, name="registro_de_decision")

## 6. Cierre verificable

**Entregable:** Protocolo de validación justificado, baseline, tabla de métricas por fold y evaluación final reservada.

- Hallazgo principal:
- Evidencia que lo respalda:
- Comparación contra baseline o escenario alternativo:
- Limitación:
- Acción, responsable y fecha de revisión:

Material elaborado por el profesor Sergio Gevatschnaider.
